# Crypto XAI audit: run the pipeline (Colab)

Arushi Nirmal, San José State University, CS 180.

This notebook installs the project, runs the tests and the quick smoke run, and shows the key
tables and figures from the full run. *Runtime → Run all* works top to bottom.

**Getting the code.** The GitHub repo may be private. Choose one:

* set `ZIP_PATH` to `crypto-xai-audit-final.zip` on your Google Drive (mount Drive first, see the next cell), **or**
* leave `ZIP_PATH` empty and the notebook tries `git clone`; if that fails (private repo), it asks you to upload the zip.

Timings measured on a laptop: quick run 16.2 s, full run 310.3 s. Colab is usually slower.

In [ ]:
import os, glob, subprocess, zipfile

REPO_URL = "https://github.com/arushiSH5-cyber/crypto-xai-audit.git"
ZIP_PATH = os.environ.get("XAI_ZIP", "")   # e.g. "/content/drive/MyDrive/crypto-xai-audit-final.zip"
MOUNT_DRIVE = False                           # set True if ZIP_PATH is on Google Drive
RUN_FULL = False                              # True = re-run the full experiment (several minutes)

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

def unpack(path):
    with zipfile.ZipFile(path) as z:
        z.extractall(".")

if not os.path.isdir("crypto-xai-audit"):
    if ZIP_PATH:
        unpack(ZIP_PATH)
    elif subprocess.run(["git", "clone", "--depth", "1", REPO_URL]).returncode != 0:
        print("git clone failed (private repo?). Upload crypto-xai-audit-final.zip:")
        from google.colab import files
        unpack(next(iter(files.upload())))
os.chdir("crypto-xai-audit")
print("working in", os.getcwd())

## Install
`lime` is installed separately. If it fails to build (Python 3.13+), the workaround from the README copies the pure-Python package in by hand.

In [ ]:
import sys
!grep -v '^lime' requirements.txt > /tmp/req_no_lime.txt && "{sys.executable}" -m pip install -q -r /tmp/req_no_lime.txt
try:
    import lime  # noqa: F401
except ImportError:
    if subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lime==0.2.0.1"]).returncode != 0:
        import site, shutil, tarfile
        subprocess.run([sys.executable, "-m", "pip", "download", "-q", "--no-deps", "--no-binary", ":all:", "lime==0.2.0.1", "-d", "/tmp/lime_src"], check=True)
        tarfile.open(glob.glob("/tmp/lime_src/lime-*.tar.gz")[0]).extractall("/tmp/lime_src")
        shutil.copytree(glob.glob("/tmp/lime_src/lime-0.2.0.1/lime")[0], os.path.join(site.getsitepackages()[0], "lime"), dirs_exist_ok=True)
import lime, shap, xgboost, lightgbm
print("lime OK; shap", shap.__version__, "xgboost", xgboost.__version__, "lightgbm", lightgbm.__version__)

## 1. Tests
No look-ahead, fold ordering, causal regime threshold, metrics, TreeSHAP determinism. The slow end-to-end reproducibility test is skipped here to save time; run `!python -m pytest -q` for all 25.

In [ ]:
!python -m pytest -q --color=no -p no:cacheprovider -m 'not slow'

## 2. Quick smoke run
2 coins (BTC, ETH), test period from 2024, smaller settings. Writes `results/quick/`.

In [ ]:
!python -W ignore run_all.py --quick

In [ ]:
import pandas as pd
from IPython.display import Image, Markdown, display
q = pd.read_csv("results/quick/tables/accuracy_by_regime.csv")
display(Markdown("**Quick run: accuracy by regime**")); display(q.round(3))

## 3. Full run (optional)
The committed `results/` already hold the full run. Set `RUN_FULL = True` in the first cell to regenerate them; the numbers should come out identical.

In [ ]:
if RUN_FULL:
    !python -W ignore run_all.py
else:
    print("Skipping the full run; showing the committed results/ below.")

## 4. Key results (full run)

In [ ]:
import json
kn = json.load(open("results/tables/key_numbers.json"))
print({k: kn[k] for k in ["n_tests", "n_significant_q05", "n_points_explained", "n_test_days_per_model", "random_baseline_top3_jaccard"]})
acc = pd.read_csv("results/tables/accuracy_by_regime.csv")
display(Markdown("**Accuracy and AUC by regime**")); display(acc[["model","regime","n","accuracy","acc_lo","acc_hi","auc","always_up_acc"]].round(3))
t = pd.read_csv("results/tables/regime_tests.csv")
display(Markdown("**Headline volatile − calm tests** (positive = less stable when volatile)"))
display(t[t.family == "headline"][["detail","metric","mean_calm","mean_volatile","diff","ci_lo","ci_hi","p_perm","q_bh","p_placebo"]].round(3))
display(Markdown("**All tests significant after Benjamini–Hochberg (q < 0.05)**"))
display(t[t.significant_q05][["family","model","detail","metric","diff","q_bh"]].round(3))

In [ ]:
for f in sorted(glob.glob("results/figures/*.png")):
    display(Markdown(f"**{os.path.basename(f)}**")); display(Image(f, width=900))